# Backtest Results

Runs the backtesting engine on historical resolved contracts and visualizes performance.

**Modes (handled automatically):**
- **Real backtest** — if `data/features/snapshots.parquet` exists, runs the model on labeled historical data
- **Dummy mode** — if no snapshots exist, falls back to synthetic outcomes from `signals/predictions.json`

Set `SNAPSHOTS_PATH` and `STARTING_BALANCE` in the config cell below, then run all.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backtest.engine import run_backtest, _SNAPSHOTS_PATH, _SENTIMENT_PATH
from backtest.metrics import compute_metrics

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

# ── Config ────────────────────────────────────────────────────────────────────
# SNAPSHOTS_PATH and SENTIMENT_PATH are auto-resolved from engine.py
# (scratch dir if it exists, otherwise local project path).
# Override here only if you need a one-off path.
SNAPSHOTS_PATH   = _SNAPSHOTS_PATH
SENTIMENT_PATH   = _SENTIMENT_PATH
TRADES_CSV       = ROOT / "logs" / "backtest_trades.csv"
STARTING_BALANCE = 100.0

In [ ]:
if SNAPSHOTS_PATH.exists():
    print(f"Running real backtest on {SNAPSHOTS_PATH} ...")
    df = run_backtest(
        features_path=SNAPSHOTS_PATH,
        sentiment_path=SENTIMENT_PATH if SENTIMENT_PATH.exists() else None,
        starting_balance=STARTING_BALANCE,
        verbose=False,
    )
    mode = "real"
else:
    print("snapshots.parquet not found — running dummy backtest from predictions.json")
    df = run_backtest(starting_balance=STARTING_BALANCE, verbose=False)
    mode = "dummy"

# Save for reference
TRADES_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(TRADES_CSV, index=False)

# Normalise types
df["won"] = df["won"].astype(bool)
if "market_category" not in df.columns or df["market_category"].isna().all():
    # Fallback: infer category from ticker prefix
    prefix = df["contract_id"].str.extract(r"KX([A-Z]+)")[0].fillna("")
    _CRYPTO  = {"BTCD", "BTC", "ETH", "SOL", "XRP", "DOGE", "ADA", "AVAX", "LINK", "BNB", "EURUSD", "USDJPY", "GBPUSD"}
    _WEATHER = {"HIGHL", "HIGHC", "HIGHN", "HIGHD", "HIGHM", "LOWTL", "LOWTC", "LOWTN", "LOWTD", "LOWTH"}
    _SPORTS  = {"NBA", "NHL", "MLB", "F1", "NBACHMP", "NHLCHMP", "NCAAMB"}
    df["market_category"] = prefix.map(
        lambda s: "crypto" if s in _CRYPTO else ("weather" if s in _WEATHER else ("sports" if s in _SPORTS else "other"))
    )

print(f"Mode: {mode} | {len(df)} trades | categories: {df['market_category'].value_counts().to_dict()}")
df.head()

## Performance Summary

In [ ]:
m = compute_metrics(df, STARTING_BALANCE)

go = m["sharpe_ratio"] > 1.0 and m["win_rate"] > 0.52 and m["max_drawdown_pct"] > -30
decision = "✅ GO for live trading" if go else "❌ Stay in dry-run"

rows = [
    ("Trades",         m["n_trades"],                        ""),
    ("Total P&L",      f"${m['total_pnl']:+.2f}",            ""),
    ("ROI",            f"{m['roi_pct']:+.2f}%",              ""),
    ("Win Rate",       f"{m['win_rate']:.1%}",               "✅" if m["win_rate"] > 0.52 else "⚠️"),
    ("Avg Edge",       f"{m['avg_edge']:.4f}",               ""),
    ("Sharpe Ratio",   f"{m['sharpe_ratio']:.3f}",           "✅" if m["sharpe_ratio"] > 1.0 else "⚠️"),
    ("Sortino Ratio",  f"{m['sortino_ratio']:.3f}",          ""),
    ("Max Drawdown",   f"${m['max_drawdown']:.2f} ({m['max_drawdown_pct']:.1f}%)", ""),
    ("Week 6 Gate",    decision,                             ""),
]
summary = pd.DataFrame(rows, columns=["Metric", "Value", "Gate"])
summary

## Cumulative P&L

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df.index, df["cumulative_pnl"], marker="o", linewidth=2, color="steelblue", label="Cumulative P&L")
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.fill_between(df.index, df["cumulative_pnl"], 0,
                where=df["cumulative_pnl"] >= 0, alpha=0.15, color="green")
ax.fill_between(df.index, df["cumulative_pnl"], 0,
                where=df["cumulative_pnl"] < 0, alpha=0.15, color="red")

ax.set_xlabel("Trade #")
ax.set_ylabel("P&L ($)")
ax.set_title("Cumulative P&L")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))
ax.legend()
plt.tight_layout()
plt.show()

## Account Balance Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

balance_series = pd.concat([
    pd.Series([STARTING_BALANCE], index=[-1]),
    df["balance"],
])
ax.plot(balance_series.index, balance_series.values, marker="o", linewidth=2, color="darkorange")
ax.axhline(STARTING_BALANCE, color="gray", linewidth=0.8, linestyle="--", label=f"Start (${STARTING_BALANCE:.0f})")

ax.set_xlabel("Trade #")
ax.set_ylabel("Balance ($)")
ax.set_title("Account Balance")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))
ax.legend()
plt.tight_layout()
plt.show()

## Per-Trade P&L

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

colors = ["green" if w else "red" for w in df["won"]]
bars = ax.bar(df.index, df["pnl"], color=colors, alpha=0.8, edgecolor="white")
ax.axhline(0, color="gray", linewidth=0.8)

labels = [c.replace("KX", "") for c in df["contract_id"]]
ax.set_xticks(df.index)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("P&L ($)")
ax.set_title("Per-Trade P&L  (green = won, red = lost)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))
plt.tight_layout()
plt.show()

## Win Rate by Category

In [ ]:
cat_stats = df.groupby("market_category").agg(
    n_trades=("won", "count"),
    win_rate=("won", "mean"),
    total_pnl=("pnl", "sum"),
).reset_index()

palette = ["steelblue", "darkorange", "mediumseagreen", "mediumpurple"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.bar(cat_stats["market_category"], cat_stats["win_rate"],
       color=palette[:len(cat_stats)], alpha=0.85)
ax.axhline(0.52, color="red", linewidth=1.2, linestyle="--", label="Gate (52%)")
ax.set_ylim(0, 1)
ax.set_ylabel("Win Rate")
ax.set_title("Win Rate by Category")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()

ax = axes[1]
pnl_colors = ["green" if v >= 0 else "red" for v in cat_stats["total_pnl"]]
ax.bar(cat_stats["market_category"], cat_stats["total_pnl"], color=pnl_colors, alpha=0.8)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_ylabel("Total P&L ($)")
ax.set_title("Total P&L by Category")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.2f"))

plt.tight_layout()
plt.show()
cat_stats

## Edge vs. P&L

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

colors = ["green" if w else "red" for w in df["won"]]
scatter = ax.scatter(df["edge"], df["pnl"], c=colors, alpha=0.8, s=80, edgecolors="white", linewidth=0.5)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.axvline(0.06, color="orange", linewidth=1, linestyle=":", label="min_edge (0.06)")

for _, row in df.iterrows():
    ax.annotate(
        row["contract_id"].replace("KX", "").split("-")[0],
        (row["edge"], row["pnl"]),
        textcoords="offset points", xytext=(5, 3), fontsize=7, alpha=0.7,
    )

ax.set_xlabel("Edge (|p_model - market_price|)")
ax.set_ylabel("P&L ($)")
ax.set_title("Edge vs. P&L per Trade")
ax.legend()
plt.tight_layout()
plt.show()

## Model Calibration: p_model vs. Outcomes

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

# Perfect calibration line
ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.4, label="Perfect calibration")

# Bucket p_model into bins and compute observed resolution rate
df["p_bin"] = pd.cut(df["p_model"], bins=np.arange(0, 1.1, 0.2), include_lowest=True)
cal = df.groupby("p_bin", observed=True)["resolved_yes"].agg(["mean", "count"]).reset_index()
bin_centers = [iv.mid for iv in cal["p_bin"]]

sc = ax.scatter(bin_centers, cal["mean"], s=cal["count"] * 40,
                color="steelblue", alpha=0.8, edgecolors="white", zorder=3, label="Observed (size = n)")
ax.plot(bin_centers, cal["mean"], color="steelblue", linewidth=1.5, alpha=0.6)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("p_model")
ax.set_ylabel("Observed resolution rate")
ax.set_title("Model Calibration")
ax.legend()
plt.tight_layout()
plt.show()